# Collect regulated-country to intermediary-country exports

This notebook collects `regulated country -> intermediary country` monthly export data from UN Comtrade.

Input:
- `data/interim/intermediary_candidates.csv`
- `data/interim/intermediary_candidates_customs_raw.csv`
- `data/interim/country_code_map.csv`

Output:
- `data/interim/comtrade_regulated_to_intermediary_plan.csv`
- `data/interim/comtrade_regulated_to_intermediary_status.csv`
- `data/interim/comtrade_regulated_to_intermediary_raw.csv`

The raw output keeps the same column layout as `intermediary_candidates_customs_raw.csv` as much as possible.
For this Comtrade file, `imp_dlr` stores export value in USD and `imp_wgt` stores export quantity/weight.


In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import comtradeapicall
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "trade-circumvention-monitor" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_comtrade_api_key

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
CANDIDATES_PATH = DATA_INTERIM / "intermediary_candidates.csv"
CUSTOMS_RAW_PATH = DATA_INTERIM / "intermediary_candidates_customs_raw.csv"
COUNTRY_CODE_MAP_PATH = DATA_INTERIM / "country_code_map.csv"

PLAN_PATH = DATA_INTERIM / "comtrade_regulated_to_intermediary_plan.csv"
STATUS_PATH = DATA_INTERIM / "comtrade_regulated_to_intermediary_status.csv"
RAW_PATH = DATA_INTERIM / "comtrade_regulated_to_intermediary_raw.csv"

CHECKPOINT_EVERY = 20

COL_PERIOD = "\uc0ac\uac74\uae30\uac04"
COL_REGULATED = "\uaddc\uc81c\uad6d"
COL_CANDIDATES = "\uc911\uac04\uad6d\ud6c4\ubcf4"


In [ ]:
RAW_COLUMNS = [
    "request_id",
    "event_id",
    COL_PERIOD,
    "hs_code",
    "original_hs_code",
    COL_REGULATED,
    "regulated_iso3",
    "start_yymm",
    "end_yymm",
    "year",
    "intermediary_country",
    "intermediary_customs_code",
    "response_hs_code",
    "imp_dlr",
    "imp_wgt",
]

STATUS_COLUMNS = ["request_id", "status", "rows", "error", "started_at", "finished_at"]


def safe_to_csv(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_csv(path, index=False, encoding="utf-8-sig")
        return path
    except PermissionError:
        fallback = path.with_name(f"{path.stem}_latest{path.suffix}")
        df.to_csv(fallback, index=False, encoding="utf-8-sig")
        return fallback


def read_csv_or_empty(path: Path, columns: list[str]) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path, dtype=str).fillna("")
    return pd.DataFrame(columns=columns)


def split_candidates(value: object) -> list[str]:
    return [part.strip() for part in str(value).split(",") if part.strip()]


def split_period_text(period_text: str) -> tuple[pd.Timestamp, pd.Timestamp]:
    start_text, end_text = str(period_text).split("~", maxsplit=1)
    return pd.to_datetime(start_text), pd.to_datetime(end_text)


def split_12_month_chunks(start: pd.Timestamp, end: pd.Timestamp) -> list[tuple[str, str]]:
    start_period = pd.Period(start, freq="M")
    end_period = pd.Period(end, freq="M")
    chunks = []
    current = start_period
    while current <= end_period:
        chunk_end = min(current + 11, end_period)
        chunks.append((current.strftime("%Y%m"), chunk_end.strftime("%Y%m")))
        current = chunk_end + 1
    return chunks


def month_periods(start_yymm: str, end_yymm: str) -> list[str]:
    return [p.strftime("%Y%m") for p in pd.period_range(start_yymm, end_yymm, freq="M")]


def yyyymm_to_year_month(value: str) -> str:
    text = str(value)
    return f"{text[:4]}.{text[4:6]}"


In [ ]:
candidates = pd.read_csv(CANDIDATES_PATH, dtype=str).fillna("")
customs_raw = pd.read_csv(CUSTOMS_RAW_PATH, dtype=str).fillna("")
country_code_map = pd.read_csv(COUNTRY_CODE_MAP_PATH, dtype=str).fillna("")

print("candidates", candidates.shape)
print("customs_raw", customs_raw.shape)
print("country_code_map", country_code_map.shape)
candidates.head()


In [ ]:
# Candidate country name -> Customs ISO2 code, learned from Korea Customs raw.
candidate_customs_code_map = (
    customs_raw[["intermediary_country", "intermediary_customs_code"]]
    .drop_duplicates()
    .query("intermediary_country != '' and intermediary_customs_code != '' and intermediary_customs_code != '-' and intermediary_country != '-'")
    .set_index("intermediary_country")["intermediary_customs_code"]
    .to_dict()
)

# Comtrade partner code by ISO2.
partner_ref = comtradeapicall.getReference("partner")
partner_ref = partner_ref[partner_ref["isGroup"].eq(False) & partner_ref["entryExpiredDate"].isna()].copy()
partner_by_iso2 = (
    partner_ref.dropna(subset=["PartnerCodeIsoAlpha2"])
    .drop_duplicates("PartnerCodeIsoAlpha2")
    .set_index("PartnerCodeIsoAlpha2")["PartnerCode"]
    .astype(str)
    .to_dict()
)

# Regulated country reporter code from country_code_map.
country_code_lookup = country_code_map.set_index("country_name_kr").to_dict("index")

print("candidate customs code map", len(candidate_customs_code_map))
print("partner iso2 map", len(partner_by_iso2))


In [ ]:
def build_collection_plan(candidates: pd.DataFrame) -> pd.DataFrame:
    rows = []
    missing = []
    for _, event in candidates.iterrows():
        regulated_country = event[COL_REGULATED]
        regulated_codes = country_code_lookup.get(regulated_country, {})
        reporter_code = str(regulated_codes.get("comtrade_reporter_code", ""))
        regulated_iso3 = str(regulated_codes.get("iso3", ""))
        if not reporter_code:
            missing.append({"event_id": event["event_id"], "country": regulated_country, "reason": "missing_reporter_code"})
            continue

        start, end = split_period_text(event[COL_PERIOD])
        chunks = split_12_month_chunks(start, end)
        for intermediary_country in split_candidates(event[COL_CANDIDATES]):
            customs_code = candidate_customs_code_map.get(intermediary_country, "")
            partner_code = ""
            if intermediary_country in country_code_lookup:
                partner_code = str(country_code_lookup[intermediary_country].get("comtrade_partner_code", ""))
            if not partner_code and customs_code:
                partner_code = str(partner_by_iso2.get(customs_code, ""))
            if not partner_code:
                missing.append({"event_id": event["event_id"], "country": intermediary_country, "reason": "missing_partner_code"})
                continue

            for start_yymm, end_yymm in chunks:
                rows.append(
                    {
                        "request_id": f"REQ-{len(rows) + 1:08d}",
                        "event_id": event["event_id"],
                        COL_PERIOD: event[COL_PERIOD],
                        "hs_code": event["hs_code"],
                        "original_hs_code": event["hs_code"],
                        COL_REGULATED: regulated_country,
                        "regulated_iso3": regulated_iso3,
                        "start_yymm": start_yymm,
                        "end_yymm": end_yymm,
                        "intermediary_country": intermediary_country,
                        "intermediary_customs_code": customs_code,
                        "reporter_code": reporter_code,
                        "partner_code": partner_code,
                    }
                )

    plan = pd.DataFrame(rows)
    missing_df = pd.DataFrame(missing).drop_duplicates() if missing else pd.DataFrame(columns=["event_id", "country", "reason"])
    return plan, missing_df


plan, missing_codes = build_collection_plan(candidates)
safe_to_csv(plan, PLAN_PATH)
safe_to_csv(missing_codes, DATA_INTERIM / "comtrade_regulated_to_intermediary_missing_codes.csv")

print("plan", plan.shape)
print("missing_codes", missing_codes.shape)
plan.head()


In [ ]:
def collect_one_request(request: pd.Series, api_key: str) -> list[dict]:
    periods = month_periods(request["start_yymm"], request["end_yymm"])
    response_df = comtradeapicall.getFinalData(
        api_key,
        typeCode="C",
        freqCode="M",
        clCode="HS",
        period=",".join(periods),
        reporterCode=str(request["reporter_code"]),
        cmdCode=str(request["hs_code"]),
        flowCode="X",
        partnerCode=str(request["partner_code"]),
        partner2Code=None,
        customsCode=None,
        motCode=None,
        maxRecords=250_000,
        format_output="JSON",
        aggregateBy=None,
        breakdownMode="classic",
        countOnly=None,
        includeDesc=True,
    )
    if response_df is None:
        raise RuntimeError("UN Comtrade returned None. Quota may be exhausted.")
    return normalize_comtrade_response(request, response_df, periods)


def normalize_comtrade_response(request: pd.Series, response_df: pd.DataFrame, periods: list[str]) -> list[dict]:
    if response_df is None or response_df.empty:
        return [empty_output_row(request, period) for period in periods]

    df = response_df.copy()
    year_col = "refYear" if "refYear" in df.columns else "period"
    month_col = "refMonth" if "refMonth" in df.columns else None
    quantity_col = next((c for c in ["qty", "netWgt", "grossWgt", "primaryQuantity"] if c in df.columns), None)
    value_col = next((c for c in ["primaryValue", "fobvalue", "cifvalue"] if c in df.columns), None)

    if month_col:
        df["period"] = df[year_col].astype(str).str.zfill(4) + df[month_col].astype(str).str.zfill(2)
    else:
        df["period"] = df[year_col].astype(str).str.replace(".", "", regex=False).str[:6]
    if quantity_col:
        df[quantity_col] = pd.to_numeric(df[quantity_col], errors="coerce").fillna(0)
    if value_col:
        df[value_col] = pd.to_numeric(df[value_col], errors="coerce").fillna(0)

    grouped = (
        df.groupby("period", dropna=False)
        .agg(
            value=(value_col, "sum") if value_col else ("period", "size"),
            quantity=(quantity_col, "sum") if quantity_col else ("period", "size"),
            response_hs_code=("cmdCode", "first") if "cmdCode" in df.columns else ("period", "first"),
        )
        .reset_index()
    )
    row_by_period = {
        str(row["period"]): output_row(
            request,
            str(row["period"]),
            response_hs_code=row["response_hs_code"],
            value=row["value"] if value_col else 0,
            quantity=row["quantity"] if quantity_col else 0,
        )
        for _, row in grouped.iterrows()
    }
    return [row_by_period.get(period, empty_output_row(request, period)) for period in periods]


def output_row(request: pd.Series, period: str, response_hs_code: object, value: object, quantity: object) -> dict:
    return {
        "request_id": request["request_id"],
        "event_id": request["event_id"],
        COL_PERIOD: request[COL_PERIOD],
        "hs_code": request["hs_code"],
        "original_hs_code": request["original_hs_code"],
        COL_REGULATED: request[COL_REGULATED],
        "regulated_iso3": request["regulated_iso3"],
        "start_yymm": request["start_yymm"],
        "end_yymm": request["end_yymm"],
        "year": yyyymm_to_year_month(period),
        "intermediary_country": request["intermediary_country"],
        "intermediary_customs_code": request["intermediary_customs_code"],
        "response_hs_code": response_hs_code,
        "imp_dlr": value,
        "imp_wgt": quantity,
    }


def empty_output_row(request: pd.Series, period: str) -> dict:
    return output_row(request, period, response_hs_code=request["hs_code"], value=0, quantity=0)


In [ ]:
api_key = get_comtrade_api_key()

status = read_csv_or_empty(STATUS_PATH, STATUS_COLUMNS)
raw = read_csv_or_empty(RAW_PATH, RAW_COLUMNS)
done = set(status.loc[status["status"].eq("ok"), "request_id"])
status_records = status.to_dict("records")
raw_records = raw.to_dict("records")

for idx, request in plan.iterrows():
    if request["request_id"] in done:
        continue
    started_at = pd.Timestamp.now().isoformat(timespec="seconds")
    try:
        rows = collect_one_request(request, api_key)
        raw_records.extend(rows)
        status_records.append(
            {
                "request_id": request["request_id"],
                "status": "ok",
                "rows": str(len(rows)),
                "error": "",
                "started_at": started_at,
                "finished_at": pd.Timestamp.now().isoformat(timespec="seconds"),
            }
        )
        done.add(request["request_id"])
    except RuntimeError as exc:
        status_records.append(
            {
                "request_id": request["request_id"],
                "status": "quota_stopped",
                "rows": "0",
                "error": str(exc),
                "started_at": started_at,
                "finished_at": pd.Timestamp.now().isoformat(timespec="seconds"),
            }
        )
        safe_to_csv(pd.DataFrame(status_records, columns=STATUS_COLUMNS), STATUS_PATH)
        safe_to_csv(pd.DataFrame(raw_records, columns=RAW_COLUMNS), RAW_PATH)
        print("Stopped, likely quota exhausted:", exc)
        break
    except Exception as exc:  # noqa: BLE001
        status_records.append(
            {
                "request_id": request["request_id"],
                "status": "error",
                "rows": "0",
                "error": repr(exc)[:500],
                "started_at": started_at,
                "finished_at": pd.Timestamp.now().isoformat(timespec="seconds"),
            }
        )

    if (idx + 1) % CHECKPOINT_EVERY == 0 or idx == len(plan) - 1:
        saved_status = safe_to_csv(pd.DataFrame(status_records, columns=STATUS_COLUMNS), STATUS_PATH)
        saved_raw = safe_to_csv(pd.DataFrame(raw_records, columns=RAW_COLUMNS), RAW_PATH)
        print(f"checkpoint {idx + 1}/{len(plan)} done={len(done)} raw_rows={len(raw_records)} status={saved_status.name} raw={saved_raw.name}")
        time.sleep(0.2)

safe_to_csv(pd.DataFrame(status_records, columns=STATUS_COLUMNS), STATUS_PATH)
safe_to_csv(pd.DataFrame(raw_records, columns=RAW_COLUMNS), RAW_PATH)
print("done", len(done), "of", len(plan), "raw_rows", len(raw_records))


In [ ]:
# UN Comtrade ??? ?? ??? ??
# - reporter_code: ??? ?? (?: ?? 156, ?? 842)
# - partner_code: ??? ?? (?: ?? 276, ???? 528, World 0)
# - cmd_code: HS ?? (?: 4412, 701912, TOTAL)
# - flow_code: X=??, M=??
# - freq_code: A=??, M=??

def fetch_un_comtrade_raw(
    reporter_code,
    partner_code,
    cmd_code,
    periods,
    flow_code="X",
    freq_code="A",
    max_records=250_000,
):
    if isinstance(periods, (list, tuple, set)):
        period_text = ",".join(str(p) for p in periods)
    else:
        period_text = str(periods)

    df = comtradeapicall.getFinalData(
        api_key,
        typeCode="C",
        freqCode=freq_code,
        clCode="HS",
        period=period_text,
        reporterCode=str(reporter_code),
        cmdCode=str(cmd_code),
        flowCode=str(flow_code),
        partnerCode=str(partner_code),
        partner2Code=None,
        customsCode=None,
        motCode=None,
        maxRecords=max_records,
        format_output="JSON",
        aggregateBy=None,
        breakdownMode="classic",
        countOnly=None,
        includeDesc=True,
    )

    print("shape:", None if df is None else df.shape)
    if df is None or df.empty:
        return df

    show_cols = [
        "period",
        "refYear",
        "refMonth",
        "reporterCode",
        "reporterDesc",
        "partnerCode",
        "partnerDesc",
        "flowCode",
        "flowDesc",
        "cmdCode",
        "cmdDesc",
        "qtyUnitAbbr",
        "qty",
        "netWgt",
        "grossWgt",
        "primaryValue",
        "fobvalue",
        "cifvalue",
    ]
    show_cols = [c for c in show_cols if c in df.columns]
    display(df[show_cols].head(30))
    return df


# Set this to True only when you want to spend API calls checking examples manually.
RUN_RAW_EXAMPLES = False

if RUN_RAW_EXAMPLES:
    sample_annual = fetch_un_comtrade_raw(
        reporter_code=156,
        partner_code=276,
        cmd_code="4412",
        periods="2013",
        flow_code="X",
        freq_code="A",
    )

    sample_monthly = fetch_un_comtrade_raw(
        reporter_code=156,
        partner_code=276,
        cmd_code="4412",
        periods="201302",
        flow_code="X",
        freq_code="M",
    )